In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 220)

df = pd.read_csv(ROOT / 'award_poster_features.csv')
df['short'] = df['name'].str.replace('★_', '', regex=False).str.replace('_poster.pdf','',regex=False).str[:24]
df.head()

## 15.1 全体の中央値・分位点 (n=15)

In [ ]:
metrics = ['text_density', 'whitespace_ratio', 'figure_area_ratio', 'figure_text_ratio',
           'section_count', 'title_h_px', 'title_area_ratio', 'edge_density',
           'colorfulness', 'contrast_L_std', 'n_text_words']
summary = df[metrics].agg(['min', 'median', 'mean', 'max', 'std']).round(3)
summary

**読み取り**

- **タイトル面積比**は中央値で全体の **約 1.4 %**（0.014）。受賞作はタイトルに大胆な面積を割いてはいない（典型的 #betterposter 形式 ≒ 20 %以上 とは別物）。
- **テキスト密度** 0.15–0.29、中央値 0.21。「ポスター面積の 5 分の 1 が文字」のオーダー。
- **ホワイトスペース比**は中央値 0.69 で、規範ガイド（30 % 推奨）より遥かに高い。これは OCR で拾えなかった**淡色背景や図の余白**まで white に算入されている可能性が高く、過大評価と解釈すべき。
- **コントラスト (L std)** と **Colorfulness** は学会間で大差なし。

## 15.2 学会間の差 — CSSJ2026 vs HPH2024

In [ ]:
g = df.groupby('conference')[metrics].agg(['median','mean']).round(3)
g.T

In [ ]:
# サンプル数が小さいので統計検定ではなく Cliff's delta（順序効果量）で大まかな方向だけ確認
def cliffs_delta(a, b):
    a = np.asarray(a); b = np.asarray(b)
    if len(a) == 0 or len(b) == 0: return np.nan
    gt = sum(x > y for x in a for y in b)
    lt = sum(x < y for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

rows = []
A = df[df.conference == 'CSSJ2026']
B = df[df.conference == 'HPH2024']
for m in metrics:
    d = cliffs_delta(A[m], B[m])
    direction = ('CSSJ > HPH' if d > 0 else ('HPH > CSSJ' if d < 0 else 'tie'))
    rows.append({'metric': m, 'cliffs_delta': round(d, 2), 'direction': direction})
pd.DataFrame(rows).sort_values('cliffs_delta', key=lambda s: s.abs(), ascending=False)

**Cliff's δ の目安**: |δ|<0.15 ≒ negligible, 0.15–0.33 small, 0.33–0.47 medium, >0.47 large

| 観察される傾向（n=15 の記述） | δ の符号 | 強さ |
|---|---|---|
| HPH のほうが **図版面積比が大きい** | 負 (大) | large |
| HPH のほうが **テキスト/ホワイトスペースが少なく、紙面が詰まっている** | 負 (大) | large |
| CSSJ のほうが **セクション数（OCR ブロック）が多い** | 正 (中) | medium |
| CSSJ のほうが **タイトル文字 px が大きい** | 正 (中) | medium |
| **Colorfulness / コントラスト / エッジ密度** はほぼ同等 | 小 | negligible |

これは「両学会のテーマ性の違い（CSSJ = 計算社会科学＝モデル・分析・テキスト中心、HPH = 健康増進＝プログラム紹介・実地写真・成果可視化中心）」を反映している可能性が高い。

## 15.3 個別の外れ値

In [ ]:
# Z スコア化して |z| > 1.5 を外れ値候補に
num = df[metrics]
z = (num - num.mean()) / num.std(ddof=0)
out_rows = []
for i, row in z.iterrows():
    for m in metrics:
        if abs(row[m]) >= 1.5:
            out_rows.append({
                'conference': df.loc[i, 'conference'],
                'short': df.loc[i, 'short'],
                'metric': m,
                'value': df.loc[i, m],
                'z': round(row[m], 2),
            })
pd.DataFrame(out_rows).sort_values(['short', 'metric'])

## 15.4 共通している（学会を跨いで揃っている）特徴

In [ ]:
# 変動係数 (std/mean) が小さい = 全件が似ている指標
cv = (num.std(ddof=0) / num.mean()).abs().sort_values()
cv.round(3).to_frame('coef_of_variation')

**変動係数が小さい順 = 受賞 15 件で揃っている指標**

- **タイトル面積比** (CV ≒ 0.4) — 大きすぎず小さすぎない約 1–2 %
- **コントラスト** (CV ≒ 0.2) — 全件 L std ≒ 40–90 の範囲
- **アスペクト比** (CV ≒ 0.1) — 14/15 件が縦長 A0 系
- **テキスト密度** (CV ≒ 0.2) — 0.15–0.29 の狭い範囲

つまり「受賞 15 件にはこの 4 項目で**揃った相場**がある」と観察できる。一方で **figure_area_ratio**, **whitespace_ratio**, **figure_text_ratio**, **section_count**, **title_h_px**, **colorfulness** は学会・作品で大きくばらつく — ここに分野依存・スタイル差が出ている。

## 15.5 散布図 — 受賞群の「形」

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = {'CSSJ2026': '#1f77b4', 'HPH2024': '#ff7f0e'}

# (1) text_density vs figure_area_ratio
ax = axes[0]
for conf, sub in df.groupby('conference'):
    ax.scatter(sub['text_density'], sub['figure_area_ratio'],
               c=colors[conf], s=120, alpha=0.75, edgecolor='white', label=conf)
    for _, r in sub.iterrows():
        ax.annotate(r['short'][:14], (r['text_density'], r['figure_area_ratio']),
                    fontsize=6, alpha=0.6, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('text_density'); ax.set_ylabel('figure_area_ratio')
ax.set_title('テキスト vs 図版'); ax.legend(); ax.grid(alpha=0.3)

# (2) colorfulness vs edge_density
ax = axes[1]
for conf, sub in df.groupby('conference'):
    ax.scatter(sub['colorfulness'], sub['edge_density'],
               c=colors[conf], s=120, alpha=0.75, edgecolor='white', label=conf)
ax.set_xlabel('colorfulness'); ax.set_ylabel('edge_density')
ax.set_title('色彩度 vs 視覚複雑性'); ax.grid(alpha=0.3)

# (3) section_count vs title_h_px
ax = axes[2]
for conf, sub in df.groupby('conference'):
    ax.scatter(sub['section_count'], sub['title_h_px'],
               c=colors[conf], s=120, alpha=0.75, edgecolor='white', label=conf)
ax.set_xlabel('section_count (OCR block + DBSCAN)'); ax.set_ylabel('title_h_px')
ax.set_title('構造の細かさ vs タイトル大きさ'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 15.6 主張できる / できないこと

### 言えそうなこと

1. **学会間でデザインプロファイルが明確に分かれている**: HPH の受賞作は「図版が多く・余白が少なく・セクション数が少なめ」、CSSJ の受賞作は「テキスト・余白が多く・セクションが細かい」。これは内容（健康現場の写真/数値 vs テキスト分析・モデル）と整合する。
2. **タイトル面積比は学会を跨いで約 1–2 %** に収まっており、「タイトルに紙面の 1/5 を割く」#betterposter 流の受賞作は **15 件中 0 件**。日本国内の主流からはまだ採用されていない様子。
3. **コントラストとアスペクト比は揃っている** — 受賞作には縦長 A0 で L std 40–90 の "業界標準" が存在。
4. **個別の外れ値（CSSJ P3-3 の Colorfulness 96 など）はテーマ表現と一致**しており、低次指標は内容の鏡として機能している。

### 言えないこと
１。非受賞のものと比較する必要がある

2. **「デザインが受賞に効いた」とは言えない** — 審査ルーブリック（§3）ではデザインは 10–20 % の重みしかなく、内容・発表力との交絡が大きい。

3. **n=15 では統計検定の出番はない**。Cliff's δ を見たのは「方向だけの目安」であり有意性の主張ではない。

### 次に進めるべきこと

| 優先 | 何をする | 効用 |
|---|---|---|
| ★★★ | **非受賞ポスターに同じパイプラインを実行** | 受賞 vs 非受賞の分布比較が初めて可能に。CSSJ ~55 件 / HPH ~325 件 |
| ★★ | **VLM (Claude/GPT-4o) でルーブリック採点を追加** | デザイン品質・main finding の伝わりやすさ等、低次指標で測れない高次特徴を補完 |
| ★★ | **`jpn_vert` 縦書き OCR を併用** | 一部 CSSJ ポスター（縦組み）でテキスト密度が過小評価されている疑いを解消 |
| ★ | **LayoutLMv3 でセクション分割を差し替え** | 現在の OCR ブロック近似より粒度の高い「タイトル / 図 / 表 / 本文」ラベル |


---

# 16. 全件比較 — 受賞ポスター vs 非受賞ポスター

§15 までは受賞 15 件の記述統計だった。ここからは **同パイプラインを全 405 件**（CSSJ2026 75 件、HPH2024 330 件）に適用した結果を読む。
各学会の中で **受賞 vs 非受賞** の分布差を確認する。



In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from scipy import stats
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 220)

ALL = pd.read_csv(ROOT / 'all_poster_features.csv')
print('Loaded', len(ALL), 'rows')
print(ALL.groupby(['conference', 'award']).size())

## 16.1 ファイル種別と受賞ラベルの内訳

In [ ]:
# HPH は jpg/pdf/png 混在。ファイル種別ごとに偏りがあるか確認
ct = ALL.groupby(['conference','ext','award']).size().unstack(fill_value=0)
ct.columns = ['non_award', 'award']
ct

## 16.2 各指標の中央値（受賞 vs 非受賞、学会別）

In [ ]:
metrics = ['text_density','whitespace_ratio','figure_area_ratio','figure_text_ratio',
           'section_count','title_h_px','title_area_ratio','edge_density',
           'colorfulness','contrast_L_std','n_text_words','aspect_ratio']

med = ALL.groupby(['conference','award'])[metrics].median().round(3)
med

## 16.3 学会内の差分検定 (Mann-Whitney U) と効果量 (Cliff's δ)

n が小さく分布も非正規が想定されるので、ノンパラメトリックで比較する。

In [ ]:
def cliffs_delta(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a)==0 or len(b)==0: return np.nan
    # O(n*m) で十分（n_max≈325）
    gt = sum((x > b).sum() for x in a)
    lt = sum((x < b).sum() for x in a)
    return (gt - lt) / (len(a) * len(b))

def compare(df, conf):
    sub = df[df.conference == conf]
    A = sub[sub.award]
    B = sub[~sub.award]
    rows = []
    for m in metrics:
        a = A[m].dropna(); b = B[m].dropna()
        if len(a) < 2 or len(b) < 2:
            rows.append({'metric': m, 'award_med': a.median() if len(a) else np.nan,
                         'nonaward_med': b.median() if len(b) else np.nan,
                         'p': np.nan, 'cliff_delta': np.nan, 'n_award': len(a), 'n_non': len(b)})
            continue
        u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
        d = cliffs_delta(a, b)
        rows.append({'metric': m,
                     'award_med': round(a.median(), 3),
                     'nonaward_med': round(b.median(), 3),
                     'diff': round(a.median() - b.median(), 3),
                     'p': round(p, 4),
                     'cliff_delta': round(d, 2),
                     'abs_d': abs(round(d,2)),
                     'n_award': len(a), 'n_non': len(b)})
    df_res = pd.DataFrame(rows).sort_values('abs_d', ascending=False).drop(columns='abs_d')
    return df_res

print('=== CSSJ2026 ===')
res_cssj = compare(ALL, 'CSSJ2026')
print(res_cssj.to_string(index=False))
print('\n=== HPH2024 ===')
res_hph = compare(ALL, 'HPH2024')
print(res_hph.to_string(index=False))

## 16.4 可視化 — 各学会で差が大きいトップ指標

In [ ]:
def boxstrip(ax, df, conf, metric, ylab=None):
    sub = df[df.conference == conf]
    parts = [sub.loc[~sub.award, metric].dropna().values,
             sub.loc[ sub.award, metric].dropna().values]
    bp = ax.boxplot(parts, positions=[0,1], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor='#eeeeee'), medianprops=dict(color='black'))
    # ストリップ
    for i, vals in enumerate(parts):
        xs = np.random.RandomState(0).normal(i, 0.06, size=len(vals))
        ax.scatter(xs, vals, alpha=0.45, s=14,
                   color='tab:orange' if i==1 else 'tab:blue', edgecolor='white', linewidth=0.5)
    ax.set_xticks([0,1]); ax.set_xticklabels(['non-award', 'award'])
    ax.set_title(f'{conf}: {metric}', fontsize=10)
    if ylab: ax.set_ylabel(ylab)
    ax.grid(axis='y', alpha=0.3)

# 各学会で |delta| が大きい 4 指標
def top_metrics(res, k=4):
    return res.dropna(subset=['cliff_delta']).head(k)['metric'].tolist()

cssj_top = top_metrics(res_cssj)
hph_top  = top_metrics(res_hph)
print('CSSJ top:', cssj_top)
print('HPH  top:', hph_top)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, m in zip(axes[0], cssj_top): boxstrip(ax, ALL, 'CSSJ2026', m)
for ax, m in zip(axes[1], hph_top):  boxstrip(ax, ALL, 'HPH2024', m)
plt.tight_layout(); plt.show()

## 16.5 共通指標 — 両学会で同じ方向に差があるもの

In [ ]:
merged = res_cssj.merge(res_hph, on='metric', suffixes=('_cssj','_hph'))
merged['same_sign'] = (np.sign(merged['cliff_delta_cssj']) == np.sign(merged['cliff_delta_hph']))
merged = merged[['metric','cliff_delta_cssj','p_cssj','cliff_delta_hph','p_hph','same_sign']]
merged.sort_values(by=['same_sign','cliff_delta_cssj'], ascending=[False, False])

## 16.6 受賞 vs 非受賞を散布図で

In [ ]:
def scatter_pair(ax, df, conf, xm, ym):
    sub = df[df.conference == conf]
    for award, color, label in [(False, '#1f77b4', 'non-award'), (True, '#ff7f0e', 'award')]:
        s = sub[sub.award == award]
        ax.scatter(s[xm], s[ym], alpha=0.55 if not award else 0.95,
                   s=20 if not award else 80, color=color, label=label,
                   edgecolor='white', linewidth=0.5)
    ax.set_xlabel(xm); ax.set_ylabel(ym); ax.set_title(conf, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
pairs = [('text_density','figure_area_ratio'),
         ('colorfulness','contrast_L_std'),
         ('section_count','title_area_ratio')]
for j,(xm,ym) in enumerate(pairs):
    scatter_pair(axes[0,j], ALL, 'CSSJ2026', xm, ym)
    scatter_pair(axes[1,j], ALL, 'HPH2024',  xm, ym)
plt.tight_layout(); plt.show()

## 16.7 多次元での要約 — PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 数値列のみ・NaN を平均で埋める
X_cols = ['text_density','whitespace_ratio','figure_area_ratio',
          'section_count','title_area_ratio','edge_density',
          'colorfulness','contrast_L_std','aspect_ratio']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, conf in zip(axes, ['CSSJ2026','HPH2024']):
    sub = ALL[ALL.conference == conf].copy()
    X = sub[X_cols].fillna(sub[X_cols].mean(numeric_only=True)).values
    if len(X) < 3: continue
    Xs = StandardScaler().fit_transform(X)
    p = PCA(n_components=2).fit(Xs)
    Z = p.transform(Xs)
    for award, color, label, size, alpha in [(False,'#1f77b4','non-award',30,0.55),
                                              (True,'#ff7f0e','award',140,0.95)]:
        m = (sub.award == award).values
        ax.scatter(Z[m,0], Z[m,1], color=color, label=label, s=size, alpha=alpha,
                   edgecolor='white', linewidth=0.6)
    ax.set_title(f'{conf} PCA  (var={p.explained_variance_ratio_.sum():.2f})')
    ax.set_xlabel(f'PC1 ({p.explained_variance_ratio_[0]*100:.0f}%)')
    ax.set_ylabel(f'PC2 ({p.explained_variance_ratio_[1]*100:.0f}%)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 16.8 ロジスティック回帰で受賞確率を予測してみる

サンプル不均衡（CSSJ 約 1:8、HPH 約 1:65）に注意しつつ、参考程度に。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

for conf in ['CSSJ2026', 'HPH2024']:
    sub = ALL[ALL.conference == conf].copy()
    if sub.award.sum() < 3:
        print(f'{conf}: 受賞 n<3 でスキップ'); continue
    X = sub[X_cols].fillna(sub[X_cols].mean(numeric_only=True)).values
    y = sub.award.astype(int).values
    Xs = StandardScaler().fit_transform(X)
    # クラス不均衡対策に class_weight='balanced'
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=0)
    clf.fit(Xs, y)
    proba = clf.predict_proba(Xs)[:,1]
    # in-sample AUC（過学習する。あくまで識別力の目安）
    auc = roc_auc_score(y, proba)
    print(f'=== {conf} ===  in-sample AUC = {auc:.3f}   (n_award={y.sum()}, n_non={(y==0).sum()})')
    coefs = pd.DataFrame({'feature': X_cols, 'coef': clf.coef_[0].round(3)})
    coefs['abs'] = coefs['coef'].abs()
    print(coefs.sort_values('abs', ascending=False).drop(columns='abs').to_string(index=False))
    print()



### 言えること

1. **CSSJ2026 で受賞ポスターは "整って密度が中程度" の傾向**
   - 中央値で見ると、受賞群の text_density は中央寄り（極端な薄/濃を避ける）、
     ホワイトスペース比は非受賞より高め？
   - Cliff's δ で |δ| ≥ 0.33 となった指標があれば、その軸でデザインが弁別的に効いていそう。
2. **HPH2024 は受賞 n=5 のため検定の解釈が難しい** が、
   - 図版面積比が高い受賞傾向（健康現場ポスターの特性と整合）。
   - 言語混在（日本語 / 英語 / 中国語）が大きな交絡。サブグループ別解析が本来必要。
3. **両学会で同方向に効く指標**（§16.5 の `same_sign=True`）があれば、それは
   学会・分野を跨いだ「学術ポスターとして受賞しやすいデザイン要因」の候補かも？

### 言えないこと

1. **データが少ないのでなんとも言えないが、デザインだけでは AUC が高くならなさそう**
2. **OCR ベースの section_count や text_density は誤差が大きい**。LayoutLMv3 や VLM での補完が必要。
3. **JPG（HPH に 149 件）と PDF（HPH に 163 件）でレンダリング解像度が違う**ため、絶対値での比較に注意がひつよう。

### 次にやるべきこと

| 何 | 効用 |
|---|---|
| **VLM（Claude/GPT-4o）でルーブリック採点** を加える | デザインの "高次品質" を測れる |
| **HPH の言語別に層化解析** | 日本語/英語/中国語の効果を切り分け |
| **クロスバリデーション付きの予測モデル** | in-sample AUC ではなく汎化性能で評価 |
| **タイトル領域を NLP で評価**（"main finding を 1 文で示しているか"） | #betterposter 形式の検出 |
